In [1]:
from bpcagent import BasicAgent, CalculatorTool, Config, ToolRegistry

In [3]:
from pathlib import Path
from dotenv import load_dotenv

project_dir = Path("/Users/elsie/Documents/个人/Github/TargetAgent/BPCAgent")
load_dotenv(project_dir / ".env", override=False)

config = Config()

registry = ToolRegistry()
calculator = CalculatorTool()
registry.register_tool(calculator)

agent = BasicAgent(
    name="Assistant",
    config=config,
    sys_prompt="你是一个简洁的助手，计算问题请调用工具。",
    tool_registry=registry,
)
# 包装实际方法，只验证执行次数，仍运行计算器原始代码。
# from unittest.mock import patch
# with patch.object(calculator, "run", wraps=calculator.run) as execution:
#     answer = agent.run("请调用计算器计算 137+55*123。")
#     assert execution.call_count > 0, "模型未实际调用计算器"
#     print(f"计算器实际执行了 {execution.call_count} 次")

result = agent.run("请调用计算器计算 137+55*123。")
if result.finished is False:
    print(result.info)
else:
    # print(result)
    print(result.output.get("text", ""))


✅ 工具 'calculate' 已注册。
✅ Agent Assistant 初始化完成，工具调用: 启用
🤖 Assistant 正在解决问题: 请调用计算器计算 137+55*123。
🧠 正在调用 deepseek-flash 模型...
✅ 大语言模型非流式响应成功:
🔧 正在执行工具 'calculate'，参数: input='137+55*123'
✅ 工具 'calculate' 执行结果: 6902
🧠 正在调用 deepseek-flash 模型...
✅ 大语言模型非流式响应成功:
137 + 55 × 123 = **6902**


## Live AgentWorkflow and optional Jev evidence assessment
这一节可以独立执行，不依赖上面的旧单 Agent 示例。请使用安装了本仓库的 Python kernel。

PatientState → Review → TargetSelection → Review → DrugSelection → Review；Review 要求修改时会重跑当前阶段。Meta 暂不运行。三个任务使用 DeepSeek Flash，Review 使用 DeepSeek V4 Pro。

**Live mode incurs API charges and sends cutoff-safe public patient data to DeepSeek.** Keys are read from config/secrets.local.json or the environment, never printed. API routing uses proxy_port in config/workflow.jsonc (28374; null for direct). Jev follows config/jev.jsonc; patient-data transmission to TypeSafe remains separately controlled by allow_patient_data. The final code cell makes one additional, document-only Jev smoke-test request using NCI text, independently of whether workflow Agents choose to call Jev. NCCN external-model access remains disabled. guideline_debug_tools enables host-issued NCI text-match target/drug fixtures with debug_only=true; their scores are not clinical ranking, eligibility, efficacy or confidence. Completion is not a validated treatment recommendation.

In [1]:
from pathlib import Path
from bpcagent.workflow import AgentWorkflow, print_workflow_report
from bpcagent.workflow.config import WorkflowConfiguration

PROJECT_DIR = Path('/home/u2021202188/BPCAgent')
MODE = 'live'  # Options: 'live' (real API) or 'demo' (offline scripted model).
REQUEST = '看下BPC数据集病人051542前100天的病例，推荐他的下一步用药。'
VERBOSE = True  # Stream console activity and print each completed/failed Agent or Review turn.
MAX_PRINT_CHARS = 4000  # Per displayed tool/context payload; None prints every character.
# Resolve a unique patient suffix; day 100 is an inclusive diagnosis-relative cutoff.
# Opt-in guideline debug scores are text matches, not validated treatment recommendations.
configuration = WorkflowConfiguration(PROJECT_DIR / 'config/workflow.jsonc')
# Shell backend comes from config/tools.jsonc: local (Docker off), sandbox (Docker on), or disabled.
options = configuration.workflow_options({'mode': MODE})
model_options = configuration.model_configs() if MODE == 'live' else {}
workflow = AgentWorkflow(**options, **model_options)
{role: {'provider': cfg.default_provider, 'model': cfg.model}
 for role, cfg in workflow.configs.items()}  # 不展示 cfg 全对象或 API key


{'PatientStateAgent': {'provider': 'deepseek', 'model': 'deepseek-flash'},
 'TargetSelectionAgent': {'provider': 'deepseek', 'model': 'deepseek-flash'},
 'DrugSelectionAgent': {'provider': 'deepseek', 'model': 'deepseek-flash'},
 'ReviewAgent': {'provider': 'deepseek', 'model': 'deepseek-v4-pro'}}

In [2]:
# API proxy routing comes from proxy_port in config/workflow.jsonc.
# Console activity and detailed turn reports are also saved to logs/console.log.
workflow_result = workflow.run(REQUEST, verbose=VERBOSE, max_print_chars=MAX_PRINT_CHARS)
log_path = Path(workflow_result['log_path'])
print({'status': workflow_result['status'], 'finished': workflow_result['finished'],
       'task_dir': workflow_result['task_dir'], 'log': str(log_path)})


Workflow started


>>> Starting turn 001: PatientStateAgent | stage=PatientStateAgent | attempt=1 | reviews_turn=None

✅ Tool 'list_guideline_sources' registered.

✅ Tool 'search_guideline' registered.

✅ Tool 'get_guideline_section' registered.

...

Context budget events:
[
  {
    "request": 1,
    "estimator": "utf8_bytes_with_framing",
    "message_count": 2,
    "reserved_output_tokens": 8192,
    "input_budget": 128000,
    "estimated_before": 15520,
    "estimated_after": 15520,
    "compacted_message_indices": [],
    "status": "unchanged"
  },
  {
    "request": 2,
    "estimator": "utf8_bytes_with_framing",
    "message_count": 5,
    "reserved_output_tokens": 8192,
    "input_budget": 128000,
    "estimated_before": 48147,
    "estimated_after": 48147,
    "compacted_message_indices": [],
    "status": "unchanged"
  },
  {
    "request": 3,
    "estimator": "utf8_bytes_with_framing",
    "message_count": 8,
    "reserved_output_tokens": 8192,
    "input_budget": 128000,
    "estimated_before": 62752,
    "estimated_after": 62752,
    "compacted_message_indices": [],
    "status": "unchanged"
  }
]

Artifacts (relative to experiment directory):
{
  "guideline_audit": [
    "006_ReviewAgent/guidelines/001_get_guideline_section.json"
  ]
}

Turn directory:
/home/u2021202188/BPCAgent/workspace/2026-10-28/run_5a914a9c3c96476e88dca639ba132b6a/006_ReviewAgent

Workflow finished: completed

{'status': 'completed', 'finished': True, 'task_dir': '/home/u2021202188/BPCAgent/workspace/2026-10-28/run_5a914a9c3c96476e88dca639ba132b6a', 'log': '/home/u2021202188/BPCAgent/workspace/2026-10-28/run_5a914a9c3c96476e88dca639ba132b6a/logs/console.log'}


In [3]:
import json

# A task Agent or Review invocation is one turn; model/tool exchanges are shown inside it.
turn_summary = [
    {'turn': turn['turn'], 'agent': turn['agent'], 'attempt': turn['attempt'],
     'status': turn['status'], 'reviews_turn': turn['reviews_turn'],
     'decision': (turn['output'] or {}).get('disposition'),
     'reason': (turn['output'] or {}).get('reason'), 'error': turn['error']}
    for turn in workflow_result['turns']
]
print(json.dumps(turn_summary, ensure_ascii=False, indent=2))
print('Workflow Jev metadata:', json.dumps(workflow_result.get('jev', {}), indent=2))
jev_assessments = [
    {'turn': turn['turn'], 'agent': turn['agent'], **assessment}
    for turn in workflow_result['turns']
    for assessment in turn.get('evidence_assessments', [])
]
print('Workflow Jev assessments:', json.dumps(jev_assessments, ensure_ascii=False, indent=2))

# Optional: inspect an earlier experiment without calling any model or tool.
# Example: PROJECT_DIR / 'workspace/2026-10-28/run_9e016013869a405e9be88a8b5425a2f2'
RUN_TO_INSPECT = None
if RUN_TO_INSPECT is not None:
    print_workflow_report(RUN_TO_INSPECT, max_chars=MAX_PRINT_CHARS)


[
  {
    "turn": 1,
    "agent": "PatientStateAgent",
    "attempt": 1,
    "status": "completed",
    "reviews_turn": null,
    "decision": null,
    "reason": null,
    "error": null
  },
  {
    "turn": 2,
    "agent": "ReviewAgent",
    "attempt": 1,
    "status": "completed",
    "reviews_turn": 1,
    "decision": "proceed",
    "reason": "PatientStateAgent 输出与共享快照在患者身份、治疗时间线、疗效证据与缺失标注上逐项一致：治疗状态(ended/unknown)如实保留未越界推断，PSA 300.9→1.01 ng/mL 与影像评估准确引用，PD-L1/MSI/MMR 正确标注为\"未报告\"而非阴性，state_check 的 complete=false 与 unknown/not_normalized/not_reported 计数均如实披露。无时间泄漏(全部事件 ≤ day 100)，未使用 cutoff 后标签，未虚构任何靶点/药物/评分或引用。本阶段正确将\"下一步用药推荐\"留给下游角色，未越权认证，符合职责分工。输出完整、安全，可进入下一阶段。",
    "error": null
  },
  {
    "turn": 3,
    "agent": "TargetSelectionAgent",
    "attempt": 1,
    "status": "completed",
    "reviews_turn": null,
    "decision": null,
    "reason": null,
    "error": null
  },
  {
    "turn": 4,
    "agent": "ReviewAgent",
    "attempt": 1,
    "status": "completed",
    "reviews_turn

### Document-only Jev live smoke test
This separate tool check sends one complete, size-bounded NCI section, never patient records or NCCN text. It tests the real provider and response validation; it does not count as a workflow Review. A high judge_confidence measures distribution concentration, not clinical correctness. Failed calls are shown as failures, with their audit retained.

In [4]:
from bpcagent.tools.guideline_library import GuidelineLibrary
from bpcagent.tools.jev import AssessEvidenceArgs, AssessEvidenceTool, JevEvaluator, JevSettings

if MODE != 'live' or not options.get('jev', {}).get('enabled'):
    print('Jev live smoke test skipped: select live mode and enable config/jev.jsonc.')
else:
    smoke_settings = JevSettings.model_validate({
        **options['jev'], 'allow_patient_data': False,
        'max_calls_per_turn': 1, 'max_calls_per_run': 1,
    })
    library = GuidelineLibrary(options['guidelines']['library_config'], external=True)
    hits = library.search('docetaxel', document_id='nci-prostate-pdq', limit=5)['results']
    sections = [library.section('nci-prostate-pdq', hit['section_id'], max_chars=12000) for hit in hits]
    complete = [section for section in sections if section['complete_section']]
    if not complete:
        raise RuntimeError('No complete bounded NCI section available; no API request sent.')
    smoke_section = min(complete, key=lambda section: len(section['text'].encode('utf-8')))
    def resolve_smoke_evidence(args):
        if args.patient_fields or args.patient_evidence_refs or args.document_refs != [smoke_section['citation_id']]:
            raise ValueError('This smoke test accepts only the selected NCI citation.')
        return {'documents': [smoke_section]}
    smoke_evaluator = JevEvaluator(smoke_settings, api_key=model_options['jev_api_key'], allow_network=True)
    smoke_artifacts = {}
    smoke_tool = AssessEvidenceTool(smoke_evaluator, resolve_smoke_evidence,
                                    Path(workflow_result['task_dir']) / 'jev_smoke', smoke_artifacts)
    smoke_result = smoke_tool.run(AssessEvidenceArgs(
        assessment_type='relevance',
        statement='Evidence about docetaxel treatment for prostate cancer; assess topic relevance, not suitability for a particular patient.',
        document_refs=[smoke_section['citation_id']],
    ))
    print(json.dumps({'result': smoke_result, 'requests_attempted': smoke_evaluator.calls,
                      'artifacts': smoke_artifacts}, ensure_ascii=False, indent=2))
    assert smoke_result['status'] == 'assessed', 'Real Jev call failed; inspect the saved result and audit.'


{
  "result": {
    "assessment_id": "jev_ea746b09908b4c8a9e64143023184e38",
    "assessment_type": "relevance",
    "status": "assessed",
    "assessment": 2.92,
    "judge_confidence": 0.92,
    "review_required": true,
    "clinical_validation": "not_performed",
    "automatic_weight_change": false,
    "rubric_version": "bpc-evidence-v1",
    "probabilities": {
      "0": 0.0,
      "1": 0.01,
      "2": 0.06,
      "3": 0.93
    },
    "answers": {
      "assessability": {
        "type": "choice",
        "choice": "assessable",
        "confidence": 0.88,
        "probabilities": {
          "assessable": 0.92,
          "insufficient_evidence": 0.07,
          "not_applicable": 0.01
        }
      },
      "assessment": {
        "type": "score",
        "score": 2.92,
        "confidence": 0.92,
        "probabilities": {
          "0": 0.0,
          "1": 0.01,
          "2": 0.06,
          "3": 0.93
        },
        "legend": {
          "0": "The evidence addresses a di

### Recorded live execution: 2026-10-28
Run: `workspace/2026-10-28/run_5a914a9c3c96476e88dca639ba132b6a`. Patient 051542, inclusive day-100 cutoff. The workflow and Jev cells above were executed with the BPCAgent .venv kernel; the legacy calculator cells were not rerun.

- **Workflow: completed, finished=true.** All six task/Review turns completed on their first attempts, with all three Reviews returning proceed. The run made 26 task/review LLM requests.
- **Debug candidates:** one androgen-signaling pathway label and one enzalutamide fixture, each with debug_only=true and debug_score=1.0. Scores indicate document text matching only; prior patient exposure does not establish future treatment suitability. Sources and limitations are included in the outputs.
- **Workflow Jev requests: 0.** Agents did not choose to call the optional evaluator; do not describe this as Jev reviewing every stage.
- **Separate document-only Jev smoke test: successful, 1 real request.** jev-1.14.0 returned relevance 2.92/3 and judge_confidence 0.92. These concern NCI document relevance, not patient reliability, drug efficacy or clinical approval. API requests used the configured 28374 proxy.
- Full turn logs, context-budget events and the Jev request audit are retained under the run directory. No patient records or NCCN text were included in the Jev smoke request. Earlier failed debugging runs and the earlier successful review/revision run remain in workspace for audit.

This is a record of this run, not a successful clinical recommendation or model-quality validation. Update this note if rerunning the notebook.